# Liu2024 - Zero-calibration cross-subject left/right MI (LOSO + Euclidean Alignment)

**Setting:** zero-calibration cross-subject. For each held-out test subject, train S-JEPA PreLocal on
the other 49 subjects and predict the held-out subject with **no labelled calibration trials** from them.

**Key addition vs the plain LOSO notebook:** per-subject **Euclidean Alignment (EA)** (He & Wu 2020).
Cross-subject MI is dominated by inter-subject distribution shift; EA re-centres each subject's trials
so every subject's mean spatial covariance is the identity, which is the single biggest cheap lever for
this setting. EA uses each subject's *unlabelled* trials only, so it stays label-free / zero-calibration.

Structure mirrors your other notebooks: Setup -> Configuration -> Data -> Alignment -> Model ->
LOSO splits -> Runner -> Run -> Aggregate.

> **REMINDER (you asked me to flag this):** run this whole LOSO pipeline once with
> `CONFIG["pretrained_mode"] = "random"` as a control. If `from_pretrained` and `random` land at the
> same accuracy, the pretrained encoder is contributing nothing on stroke MI and scaling pretraining
> won't help; if pretrained is clearly higher, the encoder transfers and pretraining is worth scaling.
> (You said you'll do this later - leaving it here so it's not lost.)

# 1. Setup

In [1]:
import os, re, sys, json, math, random, platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from scipy.io import loadmat
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import EarlyStopping
from skorch.helper import predefined_split

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal

import mne
mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("Torch", torch.__version__, "| Python", sys.version.split()[0])


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch 2.10.0 | Python 3.11.15


# 2. Configuration

## 2.1 Liu2024 channel defaults

In [2]:
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]
SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4",
    "FT7","FT8","Cz","C3","C4","T3","T4","CPz",
    "CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [n for i,n in enumerate(SOURCE_EEG_CHANNEL_NAMES_30) if i != SOURCE_REFERENCE_INDEX]


## 2.2 CONFIG

In [3]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # ---- paths / identity ----
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-zerocal-crosssubject-ea"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "zerocal_crosssubject_ea",

    # ---- dataset ----
    "subjects_to_use": None, "exclude_subjects": [],
    "source_unit": "microvolts", "final_model_unit": "microvolts",

    # ---- preprocessing ----
    "demean_mode": "none", "baseline_window_s": [0.0, 2.0],
    "reference_mode": "average", "resample": True, "resample_sfreq": 128,
    "filter_enabled": True, "filter_low": 0.5, "filter_high": 40.0,
    "filter_method": "fir", "filter_phase": "zero", "filter_fir_design": "firwin",
    "mi_window_start_s": 2.0, "target_window_samples": 537,

    # ---- ALIGNMENT (the key lever for zero-cal cross-subject) ----
    # euclidean: per-subject whiten by R^-1/2 of the mean trial covariance (label-free).
    "alignment_mode": "euclidean",        # euclidean, none
    "alignment_reg": 1e-5,                # ridge added to the reference covariance

    # ---- model ----
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",  # from_pretrained, random  (run BOTH - see reminder above)
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new", "warmup_epochs": 10,

    # ---- evaluation (LOSO, zero calibration) ----
    "evaluation_mode": "loso", "n_val_subjects": 5, "loso_test_subjects": None,

    # ---- training ----
    "batch_size": 64, "n_epochs": 5000, "early_stopping_patience": 50,
    "learning_rate": 5e-4, "optimizer_name": "adam", "weight_decay": 0.0,
    "checkpoint_metric": "valid_loss",

    "collapse_threshold": 0.90, "seed": 2026, "set_seed": True,
}


In [4]:
LIU_SOURCE_SFREQ = 500; TARGET_N_CLASSES = 2
EFFECTIVE_SFREQ = float(CONFIG["resample_sfreq"]) if CONFIG["resample"] else float(LIU_SOURCE_SFREQ)
WINDOW_SAMPLES = int(CONFIG["target_window_samples"])
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / datetime.now().strftime("%Y%m%d_%H%M")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with open(ARTIFACT_DIR / "config.json", "w") as f: json.dump(CONFIG, f, indent=2, default=str)
print(f"channels={len(EEG_CHANNEL_NAMES)} sfreq={EFFECTIVE_SFREQ} window={WINDOW_SAMPLES} "
      f"align={CONFIG['alignment_mode']} strategy={CONFIG['strategy']} pretrained={CONFIG['pretrained_mode']}")
print("artifacts:", ARTIFACT_DIR)


channels=29 sfreq=128.0 window=537 align=euclidean strategy=new pretrained=from_pretrained
artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-zerocal-crosssubject-ea/20260610_1515


## 2.3 Reproducibility and device

In [5]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True, warn_only=True)
BASE_SEED=int(CONFIG["seed"])
if CONFIG["set_seed"]: seed_everything(BASE_SEED)
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print("device", DEVICE)


device mps


# 3. Load and preprocess data

## 3.1 `.mat` loader

In [6]:
def find_source_mat_files(root):
    root=Path(root); return sorted(root.rglob("*.mat")) if root.exists() else []
def subject_id_from_path(p):
    m=re.search(r"sub[-_ ]?(\d{1,2})",str(p),re.IGNORECASE)
    if m: return int(m.group(1))
    nums=re.findall(r"\d+",Path(p).stem)
    if nums: return int(nums[-1])
    raise ValueError(p)
def _is_struct(x): return hasattr(x,"_fieldnames")
def _walk(o,pre=""):
    if isinstance(o,dict):
        for k,v in o.items():
            if str(k).startswith("__"): continue
            n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif _is_struct(o):
        for k in o._fieldnames:
            v=getattr(o,k); n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif isinstance(o,np.ndarray):
        if o.dtype==object and o.size==1: yield from _walk(o.item(),pre)
        elif o.dtype==object:
            for idx,it in np.ndenumerate(o): yield from _walk(it,f"{pre}{idx}")
def _norm_shape(a,labels=None):
    a=np.asarray(a)
    if a.ndim!=3: raise ValueError(a.shape)
    ta=[]
    if labels is not None:
        n=int(np.asarray(labels).size); ta=[ax for ax,s in enumerate(a.shape) if s==n]
    if not ta: ta=[ax for ax,s in enumerate(a.shape) if s in (39,40)]
    if ta and ta[0]!=0: a=np.moveaxis(a,ta[0],0)
    t=int(np.argmax(a.shape[1:])+1)
    if t!=2: a=np.moveaxis(a,t,2)
    if a.shape[1]<29 or a.shape[2]<1000: raise ValueError(a.shape)
    return a
def _sr(n,a):
    l=n.lower(); s=0
    if "raw" in l or "data" in l: s+=10
    if a.ndim==3: s+=5
    if any(d in (39,40) for d in a.shape): s+=3
    if max(a.shape)>=3000: s+=2
    if "eeg" in l: s+=1
    return s
def _sl(n,a):
    l=n.lower(); f=np.asarray(a).ravel(); s=0
    if "label" in l or "class" in l or l.split(".")[-1] in {"y","labels"}: s+=10
    if f.size in (39,40): s+=3
    if f.size<=200 and set(np.unique(f).astype(int)).issubset({0,1,2}): s+=2
    return s
def load_subject_mat(path):
    mat=loadmat(path,squeeze_me=True,struct_as_record=False)
    arrs=[(n,np.asarray(v)) for n,v in _walk(mat) if isinstance(v,np.ndarray) and v.dtype!=object]
    raws=[(_sr(n,a),n,a) for n,a in arrs if a.ndim==3]
    labs=[(_sl(n,a),n,a) for n,a in arrs if a.ndim in (1,2)]
    if not raws or not labs: raise KeyError(f"no data/labels in {path}")
    _,_,r=sorted(raws,key=lambda x:x[0],reverse=True)[0]
    _,_,L=sorted(labs,key=lambda x:x[0],reverse=True)[0]
    labels=np.asarray(L).astype(int).ravel()
    raw=_norm_shape(r,labels).astype(np.float64)
    if labels.size!=raw.shape[0]: raise ValueError(path)
    return raw, labels
def labels_to_zero_based(labels):
    labels=np.asarray(labels).astype(int).ravel(); u=set(np.unique(labels).tolist())
    if u.issubset({1,2}): return labels-1   # 1=left,2=right -> 0,1
    if u.issubset({0,1}): return labels
    raise ValueError(sorted(u))


## 3.2 Preprocessing (per subject)

In [7]:
def make_liu_info(sfreq):
    info=mne.create_info(EEG_CHANNEL_NAMES,float(sfreq),["eeg"]*len(EEG_CHANNEL_NAMES))
    info.set_montage(mne.channels.make_standard_montage("standard_1020"),match_case=False,on_missing="ignore")
    return info
def _to_volts(x): return np.asarray(x,np.float64)*(1e-6 if CONFIG["source_unit"].startswith("micro") else 1.0)
def _to_model(x): return np.asarray(x,np.float64)*(1e6 if CONFIG["final_model_unit"].startswith("micro") else 1.0)
def preprocess_subject(rawdata, labels):
    X=rawdata[:,EEG_CHANNEL_INDICES,:].astype(np.float64); n=X.shape[0]
    if CONFIG["demean_mode"]=="baseline_window_mean":
        b0,b1=CONFIG["baseline_window_s"]
        s0,s1=int(round(b0*LIU_SOURCE_SFREQ)),int(round(b1*LIU_SOURCE_SFREQ))
        X=X-X[:,:,s0:s1].mean(-1,keepdims=True)
    elif CONFIG["demean_mode"]=="trial_mean":
        X=X-X.mean(-1,keepdims=True)
    cont=_to_volts(X).transpose(1,0,2).reshape(len(EEG_CHANNEL_INDICES),-1)
    raw=mne.io.RawArray(cont,make_liu_info(LIU_SOURCE_SFREQ),verbose=False)
    if CONFIG["reference_mode"]=="average": raw.set_eeg_reference("average",projection=False,verbose=False)
    if CONFIG["resample"]: raw.resample(float(CONFIG["resample_sfreq"]),verbose=False)
    if CONFIG["filter_enabled"]:
        raw.filter(CONFIG["filter_low"],CONFIG["filter_high"],method="fir",
                   phase=CONFIG["filter_phase"],fir_design=CONFIG["filter_fir_design"],verbose=False)
    eff=float(raw.info["sfreq"]); data=_to_model(raw.get_data())
    per=data.shape[1]//n; data=data[:,:n*per]
    Xrs=data.reshape(len(EEG_CHANNEL_INDICES),n,per).transpose(1,0,2)
    st=int(round(CONFIG["mi_window_start_s"]*eff)); sp=st+WINDOW_SAMPLES
    if sp>Xrs.shape[-1]: raise ValueError(f"window {st}:{sp} > {Xrs.shape[-1]}")
    return Xrs[:,:,st:sp].astype(np.float32), labels_to_zero_based(labels).astype(np.int64)


# 4. Euclidean Alignment

Per subject: estimate the mean spatial covariance over that subject's trials, then whiten every trial
by its inverse square root so the subject's mean covariance becomes the identity. Label-free, so it is
valid in the zero-calibration setting (it uses the test subject's *unlabelled* trials to estimate the
reference). Set `alignment_mode="none"` to disable.

In [8]:
def alignment_reference_inv_sqrt(X, reg):
    # X: trials x C x T  ->  R^-1/2 of the mean trial covariance (C x C)
    C = X.shape[1]
    covs = np.einsum("nct,nkt->nck", X, X) / X.shape[-1]   # n x C x C
    R = covs.mean(0) + reg * np.eye(C, dtype=X.dtype)
    w, V = np.linalg.eigh(R)
    w = np.clip(w, 1e-12, None)
    return (V @ np.diag(w ** -0.5) @ V.T).astype(np.float32)

def apply_alignment(X, R_inv_sqrt):
    return np.einsum("ck,nkt->nct", R_inv_sqrt, X).astype(np.float32)

def maybe_align_subject(X):
    if CONFIG["alignment_mode"] == "none":
        return X
    if CONFIG["alignment_mode"] == "euclidean":
        R = alignment_reference_inv_sqrt(np.asarray(X, np.float32), float(CONFIG["alignment_reg"]))
        return apply_alignment(np.asarray(X, np.float32), R)
    raise ValueError(f"Unknown alignment_mode={CONFIG['alignment_mode']}")


## 4.1 Load + preprocess + align every subject once

In [ ]:
MAT_FILES=find_source_mat_files(SOURCE_EXTRACT_DIR)
if not MAT_FILES: raise FileNotFoundError(SOURCE_EXTRACT_DIR)
use=None if CONFIG["subjects_to_use"] is None else {int(s) for s in CONFIG["subjects_to_use"]}
excl={int(s) for s in CONFIG["exclude_subjects"]}
SUBJECT_DATA={}
for p in MAT_FILES:
    sid=subject_id_from_path(p)
    if (use is not None and sid not in use) or sid in excl: continue
    raw,lab=load_subject_mat(p)
    X,y=preprocess_subject(raw,lab)
    X=maybe_align_subject(X)            # per-subject EA, label-free
    SUBJECT_DATA[sid]=(X,y)
ALL_SUBJECTS=sorted(SUBJECT_DATA); N_CH=len(EEG_CHANNEL_NAMES)
print(f"loaded {len(ALL_SUBJECTS)} subjects | example X={SUBJECT_DATA[ALL_SUBJECTS[0]][0].shape} | "
      f"alignment={CONFIG['alignment_mode']}")


# 5. Model

In [ ]:
_INFO=make_liu_info(EFFECTIVE_SFREQ); CH_NAMES=list(_INFO["ch_names"]); CHS_INFO=_INFO["chs"]
NEW_LAYER_PREFIXES=("spatial_conv.","final_layer.")
def build_model():
    kw=dict(n_chans=N_CH,chs_info=CHS_INFO,n_times=WINDOW_SAMPLES,n_outputs=TARGET_N_CLASSES)
    if CONFIG["pretrained_mode"]=="from_pretrained":
        return SignalJEPA_PreLocal.from_pretrained(CONFIG["pretrained_repo_id"],**kw,strict=False)
    if CONFIG["pretrained_mode"]=="random":
        return SignalJEPA_PreLocal(**kw)
    raise ValueError(CONFIG["pretrained_mode"])
def set_trainable(model,phase):
    if phase=="full":
        for _,p in model.named_parameters(): p.requires_grad=True
    else:
        for n,p in model.named_parameters(): p.requires_grad=any(n.startswith(x) for x in NEW_LAYER_PREFIXES)
    nt=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if nt==0: raise RuntimeError("no trainable params")
    return nt
_m=build_model()
print("feature_encoder params:",sum(p.numel() for n,p in _m.named_parameters() if n.startswith("feature_encoder."))); del _m


feature_encoder params: 13840


# 6. LOSO datasets and runner

In [ ]:
class ArrayDataset(torch.utils.data.Dataset):
    def __init__(self,X,y): self.X=np.asarray(X,np.float32); self.y=np.asarray(y,np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i], int(self.y[i])
def stack(sids):
    return (np.concatenate([SUBJECT_DATA[s][0] for s in sids],0),
            np.concatenate([SUBJECT_DATA[s][1] for s in sids],0))
def build_loso_split(test_sid):
    others=[s for s in ALL_SUBJECTS if s!=test_sid]
    rng=np.random.default_rng(BASE_SEED+int(test_sid)); rng.shuffle(others)
    nval=int(CONFIG["n_val_subjects"])
    val_sids=sorted(others[:nval]); train_sids=sorted(others[nval:])
    Xtr,ytr=stack(train_sids); Xva,yva=stack(val_sids); Xte,yte=SUBJECT_DATA[test_sid]
    return (ArrayDataset(Xtr,ytr),ArrayDataset(Xva,yva),ArrayDataset(Xte,yte),yte,train_sids,val_sids)
def make_callbacks():
    mon="valid_loss" if CONFIG["checkpoint_metric"]=="valid_loss" else "valid_acc"
    return [("es",EarlyStopping(monitor=mon,lower_is_better=(mon=="valid_loss"),
            patience=int(CONFIG["early_stopping_patience"]),load_best=True))]
def build_clf(model,val_ds,max_epochs):
    opt=torch.optim.Adam if CONFIG["optimizer_name"]=="adam" else torch.optim.AdamW
    return EEGClassifier(model,criterion=torch.nn.CrossEntropyLoss,optimizer=opt,
        optimizer__weight_decay=float(CONFIG["weight_decay"]),lr=float(CONFIG["learning_rate"]),
        batch_size=int(CONFIG["batch_size"]),max_epochs=int(max_epochs),
        train_split=predefined_split(val_ds),callbacks=make_callbacks(),device=DEVICE,
        classes=list(range(TARGET_N_CLASSES)),iterator_train__shuffle=True,
        iterator_train__num_workers=0,iterator_valid__num_workers=0)
def collapse_ratio(yp):
    h=np.bincount(np.asarray(yp,int),minlength=TARGET_N_CLASSES); return float(h.max()/h.sum()) if h.sum() else 0.0
def run_one_loso(test_sid):
    if CONFIG["set_seed"]: seed_everything(BASE_SEED)
    tr,va,te,yte,trs,vas=build_loso_split(test_sid)
    model=build_model()
    if CONFIG["strategy"]=="new":
        set_trainable(model,"new"); clf=build_clf(model,va,CONFIG["n_epochs"]); clf.fit(tr,y=None)
    elif CONFIG["strategy"]=="full":
        set_trainable(model,"new"); clf=build_clf(model,va,int(CONFIG["warmup_epochs"])); clf.fit(tr,y=None)
        set_trainable(clf.module_,"full")
        clf.set_params(callbacks=make_callbacks(),max_epochs=int(CONFIG["n_epochs"]))
        clf.initialize_optimizer(); clf.initialize_callbacks(); clf.partial_fit(tr,y=None)
    else: raise ValueError(CONFIG["strategy"])
    yp=clf.predict(te)
    return {"test_subject":int(test_sid),"n_train":len(tr),"n_val":len(va),"n_test":len(te),
            "balanced_accuracy":float(balanced_accuracy_score(yte,yp)),
            "accuracy":float(accuracy_score(yte,yp)),
            "confusion":confusion_matrix(yte,yp,labels=range(TARGET_N_CLASSES)).tolist(),
            "collapse_ratio":collapse_ratio(yp),"val_subjects":vas,
            "y_true":np.asarray(yte,int).tolist(),"y_pred":np.asarray(yp,int).tolist()}


# 7. Run the LOSO loop

In [ ]:
test_subjects=CONFIG["loso_test_subjects"] or ALL_SUBJECTS
RESULTS=[]
for i,sid in enumerate(test_subjects,1):
    r=run_one_loso(sid); RESULTS.append(r)
    print(f"[{i:2d}/{len(test_subjects)}] subj {sid:2d}  BA={r['balanced_accuracy']*100:5.1f}%  "
          f"acc={r['accuracy']*100:5.1f}%  collapse={r['collapse_ratio']:.2f}  "
          f"(train={r['n_train']},val={r['n_val']},test={r['n_test']})")
with open(ARTIFACT_DIR/"loso_results.json","w") as f: json.dump(RESULTS,f,indent=2)


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.6931       0.5000        0.6935  1.0164
      2        0.6922       0.5250        0.6933  0.4490
      3        0.6913       0.5050        0.6931  0.3832
      4        0.6905       0.5150        0.6930  0.3946
      5        0.6898       0.5400        0.6927  0.3651
      6        0.6889       0.5300        0.6927  0.3805
      7        0.6878       0.5150        0.6927  0.3615
      8        0.6870       0.5050        0.6926  0.3976
      9        0.6861       0.5200        0.6926  0.4147
     10        0.6848       0.5150        0.6926  0.4060
     11        0.6837       0.5050        0.6927  0.4456
     12        0.6826       0.5050        0.6928  0.3389
     13        0.6812       0.5100        0.6929  0.4353
     14        0.6799       0.5050        0.6929  0.4413
     15        0.6788       0.5000        0.6928  0.4587
     16        0.6775       0.4

# 8. Aggregate

In [ ]:
ba=np.array([r["balanced_accuracy"] for r in RESULTS])
coll=np.array([r["collapse_ratio"]>=CONFIG["collapse_threshold"] for r in RESULTS])
yt=np.concatenate([r["y_true"] for r in RESULTS]); yp=np.concatenate([r["y_pred"] for r in RESULTS])
gba=balanced_accuracy_score(yt,yp); cm=confusion_matrix(yt,yp,labels=range(TARGET_N_CLASSES))
print("="*60)
print(f"Zero-cal cross-subject | align={CONFIG['alignment_mode']} | strategy={CONFIG['strategy']} | "
      f"pretrained={CONFIG['pretrained_mode']}")
print(f"  mean per-subject BA: {ba.mean()*100:.2f}% (SD {ba.std()*100:.2f})")
print(f"  global pooled BA:    {gba*100:.2f}%")
print(f"  collapsed subjects:  {int(coll.sum())}/{len(RESULTS)}")
print(f"  pooled confusion:\n{cm}")
print("="*60)
df=pd.DataFrame([{"subject":r["test_subject"],"balanced_acc_%":round(r["balanced_accuracy"]*100,1),
                  "acc_%":round(r["accuracy"]*100,1),"collapse":round(r["collapse_ratio"],2)} for r in RESULTS])
df=df.sort_values("subject").reset_index(drop=True); df.to_csv(ARTIFACT_DIR/"loso_per_subject.csv",index=False); df


## 8.1 What to compare
- **EA on vs off:** rerun with `alignment_mode="none"` - the EA gain is the headline here.
- **pretrained vs random** (the reminder at the top): tells you whether the encoder transfers at all.
- **new vs full** strategy.
- If EA + pretrained still sits at chance-ish, that is strong evidence the zero-calibration ceiling on
  acute-stroke left/right MI is genuinely low, and the productive next step is few-shot calibration.